This is the main notebook for the Databricks YouTube Stats project.

I started by downloading the files from the Kaggle dataset to my local machine. Then importing those files to this Databricks project.

Now I want to create data frames for each CSV file individually, perform exploratory data analysis (EDA).

List of CSV files imported:
- CAvideos.csv
- DEvideos.csv
- FRvideos.csv
- GBvideos.csv
- INvideos.csv
- JPvideos.csv
- KRvideos.csv
- MXvideos.csv
- RUvideos.csv
- USvideos.csv

List of JSON files imported:
- CA_category_id.json
- DE_category_id.json
- FR_category_id.json
- GB_category_id.json
- IN_category_id.json
- JP_category_id.json
- KR_category_id.json
- MX_category_id.json
- RU_category_id.json
- US_category_id.json

In [0]:
# Import dependencies
import json
from pyspark.sql import functions as F
from pyspark.sql.functions import *
from pyspark.sql.functions import col, lower, sum, size, lit, explode
from pyspark.sql.functions import to_date, to_timestamp

In [0]:
# Load CAvideos.csv into a Spark DataFrame
CAvideos_df = spark.read.csv(
    "/Workspace/Users/nickbartram25@gmail.com/youtube_stats_databricks/Resources/CAvideos.csv",
    header=True,
    inferSchema=True
)

display(CAvideos_df.limit(5))

In [0]:
# Load DEvideos.csv into a Spark DataFrame
DEvideos_df = spark.read.csv(
    "/Workspace/Users/nickbartram25@gmail.com/youtube_stats_databricks/Resources/DEvideos.csv",
    header=True,
    inferSchema=True
)

display(DEvideos_df.limit(5))

In [0]:
# Load FRvideos.csv into a Spark DataFrame
FRvideos_df = spark.read.csv(
    "/Workspace/Users/nickbartram25@gmail.com/youtube_stats_databricks/Resources/FRvideos.csv",
    header=True,
    inferSchema=True
)

display(FRvideos_df.limit(5))

In [0]:
# Load GBvideos.csv into a Spark DataFrame
GBvideos_df = spark.read.csv(
    "/Workspace/Users/nickbartram25@gmail.com/youtube_stats_databricks/Resources/GBvideos.csv",
    header=True,
    inferSchema=True
)

display(GBvideos_df.limit(5))

In [0]:
# Load INvideos.csv into a Spark DataFrame
INvideos_df = spark.read.csv(
    "/Workspace/Users/nickbartram25@gmail.com/youtube_stats_databricks/Resources/INvideos.csv",
    header=True,
    inferSchema=True
)

display(INvideos_df.limit(5))

In [0]:
# Load JPvideos.csv into a Spark DataFrame
JPvideos_df = spark.read.csv(
    "/Workspace/Users/nickbartram25@gmail.com/youtube_stats_databricks/Resources/JPvideos.csv",
    header=True,
    inferSchema=True
)

display(JPvideos_df.limit(5))

In [0]:
# Load KRvideos.csv into a Spark DataFrame
KRvideos_df = spark.read.csv(
    "/Workspace/Users/nickbartram25@gmail.com/youtube_stats_databricks/Resources/KRvideos.csv",
    header=True,
    inferSchema=True
)

display(KRvideos_df.limit(5))

In [0]:
# Load MXvideos.csv into a Spark DataFrame
MXvideos_df = spark.read.csv(
    "/Workspace/Users/nickbartram25@gmail.com/youtube_stats_databricks/Resources/MXvideos.csv",
    header=True,
    inferSchema=True
)

display(MXvideos_df.limit(5))

In [0]:
# Load RUvideos.csv into a Spark DataFrame
RUvideos_df = spark.read.csv(
    "/Workspace/Users/nickbartram25@gmail.com/youtube_stats_databricks/Resources/RUvideos.csv",
    header=True,
    inferSchema=True
)

display(RUvideos_df.limit(5))

In [0]:
# Load USvideos.csv into a Spark DataFrame
USvideos_df = spark.read.csv(
    "/Workspace/Users/nickbartram25@gmail.com/youtube_stats_databricks/Resources/USvideos.csv",
    header=True,
    inferSchema=True
)

display(USvideos_df.limit(5))

Data Validation Goals
- Verify schema consistency across country datasets
- Check for missing/null values
- Detect duplicate records
- Validate numeric ranges and logical consistency
- Standardize schemas prior to dataframe union

Planned ETL Steps
- Validate raw datasets
- Add country metadata
- Union datasets into canonical dataframe
- Perform transformations/cleaning
- Generate analytical outputs and visualizations

Manually inspect schemas of all dfs. 

Later we'll try automation, which works better at scale.

In [0]:
# Manually inspect schemas
CAvideos_df.printSchema()

In [0]:
# Manually inspect schemas
DEvideos_df.printSchema()

In [0]:
# Manually inspect schemas
FRvideos_df.printSchema()

In [0]:
# Manually inspect schemas
GBvideos_df.printSchema()

In [0]:
# Manually inspect schemas
INvideos_df.printSchema()

In [0]:
# Manually inspect schemas
JPvideos_df.printSchema()

In [0]:
# Manually inspect schemas
KRvideos_df.printSchema()

In [0]:
# Manually inspect schemas
MXvideos_df.printSchema()

In [0]:
# Manually inspect schemas
RUvideos_df.printSchema()

In [0]:
# Manually inspect schemas
USvideos_df.printSchema()

In [0]:
# Manually inspect schemas
print(CAvideos_df.printSchema())
print(DEvideos_df.printSchema())
print(FRvideos_df.printSchema())
print(GBvideos_df.printSchema())
print(INvideos_df.printSchema())
print(JPvideos_df.printSchema())
print(KRvideos_df.printSchema())
print(MXvideos_df.printSchema())
print(RUvideos_df.printSchema())
print(USvideos_df.printSchema())

Manually verifying the schema, even for a limited number of dfs, is prone to error.

Let's try creating a function to verify the schemas.

In [0]:
# Create a dictionary of all dfs
country_dfs = {
    "CA": CAvideos_df,
    "DE": DEvideos_df,
    "FR": FRvideos_df,
    "GB": GBvideos_df,
    "IN": INvideos_df,
    "JP": JPvideos_df,
    "KR": KRvideos_df,
    "MX": MXvideos_df,
    "RU": RUvideos_df,
    "US": USvideos_df
}

In [0]:
# View the dictionary
for country, df in country_dfs.items():
    print(f"Country: {country}")
    print(df.printSchema())

This method still requires manual inspection. Let's try to automate.

In [0]:
# Start by making a list of expected columns
expected_columns = CAvideos_df.columns

# Then check if there are mismatches
for country, df in country_dfs.items():
    
    if df.columns != expected_columns:
        print(f"Schema mismatch detected in {country}")

In [0]:
# Let's check a different way, by seeing what differences exist
for country, df in country_dfs.items():

    missing = set(expected_columns) - set(df.columns)
    extra = set(df.columns) - set(expected_columns)

    print(f"{country}")
    print(f"Missing columns: {missing}")
    print(f"Extra columns: {extra}")

The automated verification shows the df schemas are identical.

In [0]:
# Check for null missing values in CAvideos_df
CAvideos_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in CAvideos_df.columns
]).display()

Checking for negative values showed some issues with the data, possibly messy CSV parsing, bad delimiter handling, or corrupted/misaligned columnns:

Check to see if views are negative (shouldn't be)

CAvideos_df.filter(col("views") < 0).display()

[CAST_INVALID_INPUT] The value ' Stephen Curry and Channing Tatum to Amy Schumer' of the type "STRING" cannot be cast to "BIGINT" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"__lt__" was called from <command-8841710181262183>, line 2 in cell [36]



In [0]:
# Inpsect "views" column
CAvideos_df.select("views").show(20, truncate=False)

In [0]:
# Check how many rows are malformed
CAvideos_df.filter(~col("views").rlike("^[0-9]+$")).display(truncate=False)

It seems like the data is misaligned. Maybe a loading error.

Further sanity checks below.

In [0]:
# Check for null values
CAvideos_df.filter(col("video_id").isNull()).display()

There are no null values in "video_id" column, but still could be some ingestion issues.

Check for consistency again.

Reload a single CSV with stricter parsing.

In [0]:
# View a larger sample of CAvideos_df for manual inspection
# Looking for titles that look like descriptions, views with text, etc.
CAvideos_df.select("video_id", "title", "views").display(30, truncate=False)

Multi-line text fields are breaking the CSV row boundaries (likely)

Try a diagnosing with a fresh load.

In [0]:
# Load CAvideos.csv file again, allowing multilines, respect quote text blocks, handle internal quotes
# View a sample to see if it's handled correctly
df_raw = spark.read.option("header", True) \
    .option("multiLine", True) \
    .option("escape", "\"") \
    .csv("/Workspace/Users/nickbartram25@gmail.com/youtube_stats_databricks/Resources/CAvideos.csv")

df_raw.select("video_id", "title", "views").display()

In [0]:
#Check to see if views are negative (shouldn't be)
df_raw.filter(col("views") < 0).show(truncate=False)

In [0]:
# Count number of negative views
df_raw.filter(col("views") < 0).count()

The above shows there are zero rows were views are negative, a good sanity check.

In [0]:
# Check df_raw schema
df_raw.printSchema()

The schema was ingested by Spark as strings. Cast the necessasry columns to INT

In [0]:
# Cast columns to appropriate data types
df_typed = df_raw \
    .withColumn("views", col("views").cast("long")) \
    .withColumn("likes", col("likes").cast("long")) \
    .withColumn("dislikes", col("dislikes").cast("long")) \
    .withColumn("comment_count", col("comment_count").cast("long"))

df_typed.printSchema()

In [0]:
# View df
df_typed.display()

"trending_date" and "publish_time" are both clearly datetime columns

"trending_date" format: 17.15.11
"publish_time" format: 2017-11-13T15:48:57.000Z

These are different formats for different purposes. 

"trending_date" is a date.
"publish_time" is much more precise, it will require a timestamp.


In [0]:
# Cast "publish_time" to to_timestamp
df_typed = df_typed.withColumn(
    "publish_time",
    to_timestamp(col("publish_time"))
)

In [0]:
# Check data type of "publish_time"
df_typed.printSchema()

In [0]:
# View "publish_time" column
df_typed.select("publish_time").show(truncate=False)

In [0]:
# View "trending_date" column
df_typed.select("trending_date").show(truncate=False)

"trending_date" is in yy.dd.mm format

In [0]:
# Cast "trending_date" from string to to_date
df_typed = df_typed.withColumn(
    "trending_date",
    to_date(col("trending_date"), "yy.dd.MM")
)

In [0]:
# Check data type of "trending_date"
df_typed.printSchema()

In [0]:
# View "trending_date" column
df_typed.select("trending_date").show(truncate=False)

"trending_date" and "publish_time" have both been cast to appropriate data types.

Another method would be to create new columns, to avoid changing the data, then possibly removing the original columns if needed.

Check some columns ("video_error_or_removed", "ratings_disabled", "comments_disable") to see if they can be cast to Boolean.

In [0]:
# Check distinct entries of columns "video_error_or_removed"
df_typed.select("video_error_or_removed").distinct().show()


In [0]:
# Check distinct entries of columns "ratings_disabled"
df_typed.select("ratings_disabled").distinct().show()

In [0]:
# Check distinct entries of columns "comments_disabled"
df_typed.select("comments_disabled").distinct().show()

That method worked, but was not scalable.

Try to automate a little better.

In [0]:
# Create a list of the columns to test
possible_boolean_cols = ["video_error_or_removed", "ratings_disabled", "comments_disabled"]

In [0]:
# Create a loop to view them all at the same time
for c in possible_boolean_cols:
    df_typed.groupBy(c).count().show()

A function would make this process repeatable, better for pipeline workflows.

In [0]:
# Create profile_columns function for reusability
def profile_columns(df, cols):
    for c in cols:
        print(f"\n=== {c} ===")
        df.groupBy(c).count().orderBy("count", ascending=False).show()

In [0]:
# Call the function
profile_columns(df_typed, possible_boolean_cols)


After running the profile_columns function, it's clear that the columns are binary and can safely be cast to Boolean.

In [0]:
# Check data types again
df_typed.printSchema()

In [0]:
# Safely cast to Boolean
df_typed = df_typed \
    .withColumn("comments_disabled", lower(col("comments_disabled")).cast("boolean")) \
    .withColumn("ratings_disabled", lower(col("ratings_disabled")).cast("boolean")) \
    .withColumn("video_error_or_removed", lower(col("video_error_or_removed")).cast("boolean"))

In [0]:
# Check data types to verfiy cating
df_typed.printSchema()

"category_id" column should be examined.

In [0]:
# Examine "category_id" column
df_typed.groupBy("category_id").count().show()

"CA_category_id.json" likely has information about this column. Load into a df

In [0]:
# Load "CA_category_id.json" into a df
df_categories = spark.read.option("multiLine", True).json("/Workspace/Users/nickbartram25@gmail.com/youtube_stats_databricks/Resources/CA_category_id.json")

In [0]:
# View df_categories
df_categories.printSchema()

In [0]:
# Verify what we have
row = df_categories.select("items").limit(1).collect()[0]
print(type(row))
print(row)

In [0]:
# View nested items
row = df_categories.select("items").limit(1).collect()[0].asDict(recursive=True)
print(json.dumps(row, indent=2))

Finally, a view of the nested strucuture of the category id JSON file. 

Next let's check for null top level-fields

In [0]:
# Check for null
df_categories.select(
    col("etag").isNull().alias("etag_null"),
    col("kind").isNull().alias("kind_null"),
    col("items").isNull().alias("items_null")
).show()

In [0]:
# Check the size of the JSON df
df_categories.select(size("items").alias("item_count")).show()

Next, turn array into rows, flatten and then finally join.

Was enough verification, exploration done?

In [0]:
# Turn arrays into rows
df_exploded = df_categories.selectExpr("explode(items) as item")

In [0]:
# Flatten the nested structure
df_lookup = df_exploded.select(
    col("item.id").alias("category_id"),
    col("item.snippet.title").alias("category_name")
)

In [0]:
# Join the dfs (df_typed, df_lookup)
df_final = df_typed.join(df_lookup, on="category_id", how="left")

In [0]:
# View a small sample of the joined df
df_final.display(20, truncate=False)

In [0]:
# Double check the join worked
df_final.select("category_id", "category_name").distinct().show()

In case we create a large df for the complete dataset, a region column should be created for each CSV/df

In [0]:
# Create a region column for entire df
df_final = df_final.withColumn("region", lit("CA"))

In [0]:
df_final.display()

This df is silver by now, to make it gold remove some unnecesary columns

In [0]:
# Create the final columns for the df
df_gold = df_final.select(
    "video_id",
    "title",
    "channel_title",
    "category_name",
    "publish_time",
    "trending_date",
    "views",
    "likes",
    "dislikes",
    "comment_count",
    "comments_disabled",
    "ratings_disabled",
    "video_error_or_removed",
    "region"
)

In [0]:
# Add a like_ratio column
df_gold = df_gold.withColumn(
    "like_ratio",
    col("likes") / col("views")
)

In [0]:
# Displap df_gold
df_gold.display()

In [0]:
# Rearrange some of the columns for readability
df_gold = df_gold.select(
    "video_id",
    "title",
    "channel_title",
    "category_name",
    "publish_time",
    "trending_date",
    "views",
    "likes",
    "dislikes",
    "like_ratio",
    "comment_count",
    "comments_disabled",
    "ratings_disabled",
    "video_error_or_removed",
    "region"
)

In [0]:
df_gold.display()

Do some validation checks, perhaps should have been done before join?

Read ETL PDF from Databricks. Might make sense to do validation checks before and after? Maybe just after?

Check all columns for null

Check numeric values for impossible values (did some already when checked "views" column for negataive)

Likes should not exceed views
Dislikes should not exceed views
Comments should not exceed views

Trending date should be after publishing date

Check for duplicates

Check profiling describe(), to expose strange mac values or unexpected distributions,

In [0]:
# Check all columns for null values
df_gold.select([
    count(when(col(c).isNull(), c)). alias(c)
    for c in df_gold.columns
]).show()

There are 74 null values in category_name, other columns have zero.

In [0]:
# Check numeric columns for impossible values
# Create list of numeric values
numeric_cols = [
    "views",
    "likes",
    "dislikes",
    "comment_count"
]

# Loop through numeric columns to check for negative numbers
for c in numeric_cols:
    count_neg = df_gold.filter(col(c) < 0).count()

    print(f"{c}: {count_neg}")

There are no negative numeric values or impossible numeric values.

Check for logical consistency.

In [0]:
# Likes should not exceed views
df_typed.filter(
    col("likes") > col("views")
).count()

In [0]:
# Dislikes should not exceed views
df_typed.filter(
    col("dislikes") > col("views")
).count()

In [0]:
# Comments should not exceed views
df_typed.filter(
    col("comment_count") > col("views")
).count()

In [0]:
# Publish date after trending date?
df_typed.filter(
    col("publish_time") > col("trending_date")
).count()

In [0]:
# Check number of rows in df_gold
df_gold.count()

In [0]:
# Inspect the gold table
df_gold.select(
    "trending_date",
    "publish_time"
).show(10, truncate=False)

In [0]:
# Inspect the df_raw times
df_raw.select(
    "trending_date",
    "publish_time"
).show(10, truncate=False)

Error in formatting, should be yy.dd.MM not yy.dd.mm (MM for months, mm for minutes).

This issue was corrected in Cell 56.

Still an issue with 2031 rows containing a publish time greater than the trending time.

In [0]:
# Look at some of the bad rows
df_gold.filter(col("publish_time") > col("trending_date")) \
    .select("video_id", "publish_time", "trending_date") \
    .show(20, truncate=False)

It looks like Spark is handling the "trending_date" as 00:00:00, so anything later than that on the same day in "publish_time" is greater.

Validation needs to be more precise.


In [0]:
# Publish date after trending date?
df_gold.filter(
    to_date(col("publish_time")) > col("trending_date")
).count()

Excellent, there are no longer any publish times greater than trending dates. The sanity check is successful.

Check for duplicates.

In [0]:
# Check for duplicate rows
df_gold.groupBy(
    "video_id",
    "trending_date",
    "region"
).count().filter(
    col("count") > 1
).show()

There are no duplicate rows.

Check that the category names all mapped properly.

In [0]:
# Check for null category names
df_gold.filter(
    col("category_name").isNull()
).count()

We have 74 category_names that are null, but we did a null check before and there were zero?

In [0]:
# Inspect null rows
df_final.filter(col("category_name").isNull()) \
    .select("category_id") \
    .distinct() \
    .show()

In [0]:
# View df_gold, manually inspect nulls
df_gold.display()

In [0]:
# View df_final, manually inspect nulls
df_final.display()

Manual inspection showed that category 29 is the null values and the the US JSON ("US_category_id.json") contains this category.

Manual inspection is not scalable or precise, a better solution is to flatten all JSONs, union them, create a global table, and resolve conflicts.